In [1]:
import os,sys,time
print("=== QUEUEWATCH V5.1 BOOT ===", flush=True)
print("[BOOT] Python:", sys.version, flush=True)
print("[BOOT] cwd:", os.getcwd(), flush=True)
print("[BOOT] heartbeat 1", flush=True); time.sleep(1)
print("[BOOT] heartbeat 2", flush=True)
open("/kaggle/working/QUEUEWATCH_V5_1_STARTED.txt","w").write("started\n")
print("=== QUEUEWATCH V5.1 HEARTBEAT OK ===", flush=True)

=== QUEUEWATCH V5.1 BOOT ===
[BOOT] Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
[BOOT] cwd: /kaggle/working
[BOOT] heartbeat 1
[BOOT] heartbeat 2
=== QUEUEWATCH V5.1 HEARTBEAT OK ===


# 9% — QueueWatch v5.1 | Pretrained 5-Fold Count Regression

v4 selesai secara pipeline tetapi baseline density-regression dari nol masih lemah.  
v5 mengganti strategi menjadi **ImageNet-pretrained MobileNetV3-Small + log-count regression + 5-fold cross-validation**.

Tujuan:
- seluruh 100 labeled image ikut dalam 5-fold CV;
- evaluasi OOF MAE/RMSE yang lebih stabil daripada satu split 80/20;
- transfer learning, bukan belajar visual feature dari nol;
- ensemble 5 fold untuk 216 official test image;
- buat `submission.csv`, model fold, reports, plots, dan ZIP.

Dataset tetap **general crowd counting**, bukan queue-specific ROI dataset.

In [2]:
from pathlib import Path
ROOT=Path("/kaggle/input/datasets/wafabila/ioai-computer-vision")
print("[PREFLIGHT] exists:",ROOT.exists(),flush=True)
if not ROOT.exists(): raise FileNotFoundError(ROOT)
print("[PREFLIGHT] top-level:",sorted(p.name for p in ROOT.iterdir()),flush=True)
for name in ["labeled","test","ground_truth_train.json","sample_submission.csv"]:
    if not (ROOT/name).exists(): raise FileNotFoundError(ROOT/name)
print("[PREFLIGHT] dataset mount OK",flush=True)

[PREFLIGHT] exists: True
[PREFLIGHT] top-level: ['ground_truth_train.json', 'labeled', 'sample_submission.csv', 'test']
[PREFLIGHT] dataset mount OK


In [3]:
import gc,json,math,random,shutil,time,warnings
warnings.filterwarnings("ignore")
from pathlib import Path
def log(x): print(f"[{time.strftime('%H:%M:%S')}] {x}",flush=True)

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image
import torch, torch.nn as nn
from torch.utils.data import Dataset,DataLoader
from torchvision import transforms
from torchvision.models import mobilenet_v3_small,MobileNet_V3_Small_Weights
from sklearn.model_selection import StratifiedKFold,KFold
from sklearn.metrics import mean_absolute_error,mean_squared_error

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

DATA_ROOT=ROOT; LAB=ROOT/"labeled"; TEST=ROOT/"test"; GT=ROOT/"ground_truth_train.json"; SAMPLE=ROOT/"sample_submission.csv"
OUT=Path("/kaggle/working/queuewatch_v5_1_output"); REP=OUT/"reports"; MOD=OUT/"models"; PLOT=OUT/"plots"
for p in [OUT,REP,MOD,PLOT]: p.mkdir(parents=True,exist_ok=True)

IMG=320; BATCH=8; FOLDS=5; HEAD_EPOCHS=4; FT_EPOCHS=18; PATIENCE=5
HEAD_LR=8e-4; FT_LR=1e-4; WD=1e-4; WORKERS=2
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu"); AMP=torch.cuda.is_available()
log(f"[1/10] device={DEVICE}, gpu_count={torch.cuda.device_count() if torch.cuda.is_available() else 0}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()): log(f"[1/10] GPU {i}: {torch.cuda.get_device_name(i)}")

[22:01:13] [1/10] device=cuda, gpu_count=2
[22:01:13] [1/10] GPU 0: Tesla T4
[22:01:13] [1/10] GPU 1: Tesla T4


In [4]:
log("[2/10] dataset audit")
EXT={".jpg",".jpeg",".png",".bmp",".webp"}
train_images=sorted([p for p in LAB.rglob("*") if p.is_file() and p.suffix.lower() in EXT],key=lambda p:p.name.lower())
test_images=sorted([p for p in TEST.rglob("*") if p.is_file() and p.suffix.lower() in EXT],key=lambda p:p.name.lower())
raw=json.load(open(GT,"r",encoding="utf-8"))

def parse_counts(obj):
    out={}
    if isinstance(obj,dict):
        if obj and all(isinstance(v,(int,float,np.integer,np.floating)) for v in obj.values()):
            return {Path(str(k)).name:int(v) for k,v in obj.items()}
        for v in obj.values():
            try: out.update(parse_counts(v))
            except: pass
    elif isinstance(obj,list):
        for row in obj:
            if not isinstance(row,dict): continue
            name=row.get("image_name") or row.get("filename") or row.get("image") or row.get("file_name") or row.get("id")
            cnt=None
            for k in ["count","people_count","person_count"]:
                if k in row: cnt=row[k]; break
            if name is not None and cnt is not None: out[Path(str(name)).name]=int(cnt)
            else:
                try: out.update(parse_counts(row))
                except: pass
    return out

count_map=parse_counts(raw)
rows=[]
for p in train_images:
    c=count_map.get(p.name,count_map.get(p.stem))
    if c is None: raise RuntimeError(f"No label for {p.name}")
    rows.append({"image_name":p.name,"image_path":str(p),"count":int(c)})
manifest=pd.DataFrame(rows)
manifest.to_csv(REP/"manifest.csv",index=False)
log(f"[2/10] labeled={len(manifest)} test={len(test_images)} count[min/median/mean/max]={manifest['count'].min()}/{manifest['count'].median():.1f}/{manifest['count'].mean():.2f}/{manifest['count'].max()}")
json.dump({"labeled_images":len(manifest),"test_images":len(test_images),"count_min":int(manifest['count'].min()),"count_max":int(manifest['count'].max()),"count_mean":float(manifest['count'].mean()),"count_median":float(manifest['count'].median())},open(REP/"dataset_audit.json","w"),indent=2)

[22:01:13] [2/10] dataset audit
[22:01:14] [2/10] labeled=100 test=216 count[min/median/mean/max]=15/89.5/126.36/539


In [5]:
log("[3/10] create 5 folds")
try:
    bins=pd.qcut(manifest["count"],q=5,labels=False,duplicates="drop")
    if bins.nunique()<2: raise ValueError("insufficient bins")
    splitter=StratifiedKFold(n_splits=FOLDS,shuffle=True,random_state=SEED)
    iterator=splitter.split(manifest,bins); split_name=f"StratifiedKFold({bins.nunique()} bins)"
except Exception as e:
    log(f"[3/10][WARN] fallback KFold: {e}")
    splitter=KFold(n_splits=FOLDS,shuffle=True,random_state=SEED); iterator=splitter.split(manifest); split_name="KFold"

manifest["fold"]=-1
splits=[]
for f,(tr,va) in enumerate(iterator):
    manifest.loc[va,"fold"]=f; splits.append((np.asarray(tr),np.asarray(va)))
manifest.to_csv(REP/"manifest_with_folds.csv",index=False)
log(f"[3/10] {split_name}")
for f in range(FOLDS):
    d=manifest[manifest.fold==f]
    log(f"[3/10] fold={f} val={len(d)} mean={d['count'].mean():.2f} min={d['count'].min()} max={d['count'].max()}")

[22:01:14] [3/10] create 5 folds
[22:01:14] [3/10] StratifiedKFold(5 bins)
[22:01:14] [3/10] fold=0 val=20 mean=131.95 min=26 max=539
[22:01:14] [3/10] fold=1 val=20 mean=124.10 min=15 max=412
[22:01:14] [3/10] fold=2 val=20 mean=141.65 min=23 max=513
[22:01:14] [3/10] fold=3 val=20 mean=120.35 min=23 max=300
[22:01:14] [3/10] fold=4 val=20 mean=113.75 min=28 max=261


In [6]:
log("[4/10] transforms/dataset")
weights=MobileNet_V3_Small_Weights.DEFAULT
mean,std=weights.transforms().mean,weights.transforms().std
train_tf=transforms.Compose([transforms.Resize((IMG,IMG)),transforms.RandomHorizontalFlip(0.5),transforms.ColorJitter(0.15,0.15,0.10,0.02),transforms.ToTensor(),transforms.Normalize(mean,std)])
eval_tf=transforms.Compose([transforms.Resize((IMG,IMG)),transforms.ToTensor(),transforms.Normalize(mean,std)])

class CountDS(Dataset):
    def __init__(self,df,tf,inference=False): self.df=df.reset_index(drop=True); self.tf=tf; self.inference=inference
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]; im=self.tf(Image.open(r.image_path).convert("RGB"))
        x={"image":im,"image_name":r.image_name}
        if not self.inference:
            c=float(r["count"]); x["count"]=torch.tensor(c,dtype=torch.float32); x["log_count"]=torch.tensor(np.log1p(c),dtype=torch.float32)
        return x

[22:01:14] [4/10] transforms/dataset


In [7]:
log("[5/10] model helpers")

def build_model():
    """
    Build a MobileNetV3-Small regressor.

    IMPORTANT:
    torchvision MobileNetV3-Small produces a 576-d feature vector
    before the classifier. The previous v5 replaced the whole classifier
    using classifier[-1].in_features (=1024), which caused:
        mat1 and mat2 shapes cannot be multiplied (8x576 and 1024x256)

    Fix:
    keep the pretrained classifier trunk and only replace its final
    classification layer with a single regression output.
    """
    pretrained = True

    try:
        m = mobilenet_v3_small(weights=weights)
        log("[5/10] pretrained weights loaded")
    except Exception as e:
        pretrained = False
        log(f"[5/10][WARN] pretrained unavailable: {repr(e)}")
        m = mobilenet_v3_small(weights=None)

    # MobileNetV3-Small classifier is typically:
    # Linear(576 -> 1024) -> Hardswish -> Dropout -> Linear(1024 -> 1000)
    # Keep the pretrained 576->1024 projection and replace only the last layer.
    regression_in_features = m.classifier[-1].in_features
    m.classifier[-1] = nn.Linear(regression_in_features, 1)

    return m, pretrained


def freeze_backbone(m):
    # Stage 1: train classifier only.
    for p in m.features.parameters():
        p.requires_grad = False

    for p in m.classifier.parameters():
        p.requires_grad = True


def unfreeze_tail(m):
    # Stage 2: fine-tune only the final feature blocks + classifier.
    for p in m.features.parameters():
        p.requires_grad = False

    for block in m.features[-4:]:
        for p in block.parameters():
            p.requires_grad = True

    for p in m.classifier.parameters():
        p.requires_grad = True


def optimizer_for(m, lr):
    trainable = [
        p
        for p in m.parameters()
        if p.requires_grad
    ]

    return torch.optim.AdamW(
        trainable,
        lr=lr,
        weight_decay=WD,
    )


loss_fn = nn.SmoothL1Loss(beta=0.20)


def to_count(log_pred):
    return np.maximum(
        np.expm1(log_pred),
        0.0,
    )


@torch.no_grad()
def infer(m, loader):
    m.eval()

    names = []
    truths = []
    preds = []

    for b in loader:
        images = b["image"].to(
            DEVICE,
            non_blocking=True,
        )

        y = (
            m(images)
            .squeeze(1)
            .detach()
            .cpu()
            .numpy()
        )

        p = to_count(y)

        names.extend(b["image_name"])
        preds.extend(p.tolist())

        if "count" in b:
            truths.extend(
                b["count"].numpy().tolist()
            )

    return (
        names,
        np.asarray(truths, dtype=np.float32),
        np.asarray(preds, dtype=np.float32),
    )


# ------------------------------------------------------------
# DRY-RUN SHAPE CHECK BEFORE 5-FOLD TRAINING
# ------------------------------------------------------------
shape_check_model, shape_check_pretrained = build_model()
shape_check_model = shape_check_model.to(DEVICE)

with torch.no_grad():
    dummy = torch.zeros(
        2,
        3,
        IMG,
        IMG,
        device=DEVICE,
    )

    dummy_out = shape_check_model(dummy)

log(
    f"[5/10] dry-run input={tuple(dummy.shape)} "
    f"output={tuple(dummy_out.shape)}"
)

assert dummy_out.shape == (2, 1), (
    f"Unexpected regression output shape: {tuple(dummy_out.shape)}"
)

del shape_check_model, dummy, dummy_out
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

log("[5/10] model helpers + shape check OK")

[22:01:14] [5/10] model helpers
Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 97.1MB/s]

[22:01:15] [5/10] pretrained weights loaded


[22:01:16] [5/10] dry-run input=(2, 3, 320, 320) output=(2, 1)
[22:01:16] [5/10] model helpers + shape check OK


In [8]:
log("[6/10] 5-fold CV START")
oof=np.zeros(len(manifest),dtype=np.float32); fold_test=[]; history=[]
test_df=pd.DataFrame({"image_name":[p.name for p in test_images],"image_path":[str(p) for p in test_images]})
test_loader=DataLoader(CountDS(test_df,eval_tf,True),batch_size=BATCH,shuffle=False,num_workers=WORKERS,pin_memory=AMP)

for fold in range(FOLDS):
    start=time.time()
    trdf=manifest[manifest.fold!=fold].reset_index(drop=True)
    vadf=manifest[manifest.fold==fold].reset_index().rename(columns={"index":"orig_idx"})
    trloader=DataLoader(CountDS(trdf,train_tf),batch_size=BATCH,shuffle=True,num_workers=WORKERS,pin_memory=AMP)
    valoader=DataLoader(CountDS(vadf,eval_tf),batch_size=BATCH,shuffle=False,num_workers=WORKERS,pin_memory=AMP)
    m,pretrained_ok=build_model(); m=m.to(DEVICE)
    path=MOD/f"queuewatch_fold{fold}_best.pt"
    best=float("inf"); patience=0; global_epoch=0
    scaler=torch.cuda.amp.GradScaler(enabled=AMP)

    for stage,epochs,lr in [("head",HEAD_EPOCHS,HEAD_LR),("finetune",FT_EPOCHS,FT_LR)]:
        freeze_backbone(m) if stage=="head" else unfreeze_tail(m)
        opt=optimizer_for(m,lr)
        if stage=="finetune": patience=0
        for ep in range(1,epochs+1):
            global_epoch+=1; m.train(); running=0.0
            for bi,b in enumerate(trloader,1):
                imgs=b["image"].to(DEVICE,non_blocking=True); tgt=b["log_count"].to(DEVICE,non_blocking=True)
                opt.zero_grad(set_to_none=True)
                with torch.autocast(device_type="cuda",dtype=torch.float16,enabled=AMP):
                    outp=m(imgs).squeeze(1); loss=loss_fn(outp,tgt)
                scaler.scale(loss).backward(); scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(m.parameters(),5.0); scaler.step(opt); scaler.update()
                running+=float(loss.item())
            _,vt,vp=infer(m,valoader)
            mae=mean_absolute_error(vt,vp); rmse=math.sqrt(mean_squared_error(vt,vp))
            history.append({"fold":fold,"global_epoch":global_epoch,"stage":stage,"stage_epoch":ep,"train_loss":running/max(len(trloader),1),"val_mae":float(mae),"val_rmse":float(rmse)})
            pd.DataFrame(history).to_csv(REP/"cv_training_history.csv",index=False)
            log(f"[6/10][F{fold+1}] {stage} E{ep:02d}/{epochs:02d} | MAE={mae:.3f} RMSE={rmse:.3f}")
            if mae<best:
                best=float(mae); patience=0
                torch.save({"model_state_dict":m.state_dict(),"fold":fold,"best_val_mae":best,"img_size":IMG,"pretrained_loaded":pretrained_ok,"architecture":"MobileNetV3-Small count regression","target":"log1p(count)"},path)
            else:
                patience+=1
                if stage=="finetune" and patience>=PATIENCE:
                    log(f"[6/10][F{fold+1}] early stopping"); break

    ck=torch.load(path,map_location=DEVICE); m.load_state_dict(ck["model_state_dict"])
    _,vt,vp=infer(m,valoader); oof[vadf.orig_idx.to_numpy()]=vp
    test_names,_,tp=infer(m,test_loader); fold_test.append(tp)
    log(f"[6/10][FOLD {fold+1}] DONE | MAE={mean_absolute_error(vt,vp):.3f} | time={time.time()-start:.1f}s")
    del m,trloader,valoader; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
log("[6/10] CV COMPLETE")

[22:01:16] [6/10] 5-fold CV START
[22:01:16] [5/10] pretrained weights loaded
[22:01:22] [6/10][F1] head E01/04 | MAE=1095.812 RMSE=1286.145
[22:01:24] [6/10][F1] head E02/04 | MAE=88.038 RMSE=147.012
[22:01:25] [6/10][F1] head E03/04 | MAE=94.716 RMSE=123.413
[22:01:26] [6/10][F1] head E04/04 | MAE=73.037 RMSE=124.699
[22:01:29] [6/10][F1] finetune E01/18 | MAE=72.273 RMSE=129.130
[22:01:30] [6/10][F1] finetune E02/18 | MAE=61.418 RMSE=118.084
[22:01:32] [6/10][F1] finetune E03/18 | MAE=60.743 RMSE=114.673
[22:01:33] [6/10][F1] finetune E04/18 | MAE=60.508 RMSE=113.988
[22:01:35] [6/10][F1] finetune E05/18 | MAE=60.445 RMSE=114.125
[22:01:36] [6/10][F1] finetune E06/18 | MAE=56.197 RMSE=110.519
[22:01:38] [6/10][F1] finetune E07/18 | MAE=54.035 RMSE=108.991
[22:01:39] [6/10][F1] finetune E08/18 | MAE=65.102 RMSE=102.040
[22:01:41] [6/10][F1] finetune E09/18 | MAE=49.726 RMSE=103.468
[22:01:42] [6/10][F1] finetune E10/18 | MAE=56.105 RMSE=99.877
[22:01:44] [6/10][F1] finetune E11/18 | 

In [9]:
log("[7/10] OOF evaluation")
truth=manifest["count"].to_numpy(dtype=np.float32)
oof_mae=mean_absolute_error(truth,oof); oof_rmse=math.sqrt(mean_squared_error(truth,oof))
baseline=np.zeros(len(manifest),dtype=np.float32)
for f in range(FOLDS):
    med=float(np.median(manifest[manifest.fold!=f]["count"].to_numpy()))
    baseline[manifest.fold.to_numpy()==f]=med
base_mae=mean_absolute_error(truth,baseline); base_rmse=math.sqrt(mean_squared_error(truth,baseline))
oof_df=manifest[["image_name","count","fold"]].copy(); oof_df["predicted_count"]=oof; oof_df["absolute_error"]=(oof_df["count"]-oof_df["predicted_count"]).abs(); oof_df["baseline_prediction"]=baseline
oof_df.to_csv(REP/"oof_predictions.csv",index=False)
metrics={"oof_mae":float(oof_mae),"oof_rmse":float(oof_rmse),"naive_fold_median_mae":float(base_mae),"naive_fold_median_rmse":float(base_rmse),"mae_improvement_vs_baseline":float(base_mae-oof_mae)}
json.dump(metrics,open(REP/"cv_metrics.json","w"),indent=2)
log(f"[7/10] OOF MAE={oof_mae:.3f} RMSE={oof_rmse:.3f}")
log(f"[7/10] naive MAE={base_mae:.3f}; improvement={base_mae-oof_mae:.3f}")

[22:03:52] [7/10] OOF evaluation
[22:03:52] [7/10] OOF MAE=41.805 RMSE=65.514
[22:03:52] [7/10] naive MAE=74.260; improvement=32.455


In [10]:
log("[8/10] plots")
plt.figure(figsize=(7,7)); plt.scatter(truth,oof,alpha=.8)
mn=min(float(truth.min()),float(oof.min())); mx=max(float(truth.max()),float(oof.max())); plt.plot([mn,mx],[mn,mx],"--")
plt.xlabel("Ground Truth Count"); plt.ylabel("OOF Predicted Count"); plt.title("QueueWatch v5.1 — 5-Fold OOF"); plt.tight_layout(); plt.savefig(PLOT/"oof_scatter.png",dpi=160); plt.close()
plt.figure(figsize=(10,5)); plt.hist(oof_df.absolute_error,bins=20); plt.xlabel("Absolute Count Error"); plt.ylabel("Images"); plt.title("OOF Error Distribution"); plt.tight_layout(); plt.savefig(PLOT/"oof_error_hist.png",dpi=160); plt.close()
h=pd.read_csv(REP/"cv_training_history.csv"); plt.figure(figsize=(11,6))
for f in range(FOLDS):
    d=h[h.fold==f]; plt.plot(d.global_epoch,d.val_mae,marker="o",label=f"Fold {f+1}")
plt.xlabel("Epoch"); plt.ylabel("Validation MAE"); plt.title("Validation MAE by Fold"); plt.legend(); plt.tight_layout(); plt.savefig(PLOT/"fold_validation_mae.png",dpi=160); plt.close()

[22:03:52] [8/10] plots


In [11]:
log("[9/10] test ensemble")
mat=np.vstack(fold_test); mean_pred=mat.mean(0); median_pred=np.median(mat,axis=0); final=np.maximum(np.rint(median_pred),0).astype(int)
test_pred=pd.DataFrame({"image_name":test_names,"ensemble_mean":mean_pred,"ensemble_median":median_pred,"final_count":final})
for f in range(FOLDS): test_pred[f"fold_{f}_prediction"]=mat[f]
test_pred.to_csv(REP/"test_ensemble_predictions.csv",index=False)

sub=pd.read_csv(SAMPLE); fname=next((c for c in ["image_name","filename","image","file_name","id"] if c in sub.columns),None)
countcol=next((c for c in ["count","people_count","person_count","prediction","predicted_count"] if c in sub.columns),None)
mapping=dict(zip(test_pred.image_name,test_pred.final_count))
if fname is not None:
    if countcol is None: countcol="count"
    sub[countcol]=sub[fname].map(lambda x:mapping.get(Path(str(x)).name))
else:
    if len(sub)!=len(test_pred): raise RuntimeError("Cannot map sample submission")
    if countcol is None: countcol=sub.columns[-1] if len(sub.columns)>=2 else "count"
    sub[countcol]=test_pred.final_count.to_numpy()
if sub[countcol].isna().any(): raise RuntimeError("Missing submission predictions")
sub[countcol]=sub[countcol].round().astype(int)
sub.to_csv(OUT/"submission.csv",index=False)
log(f"[9/10] submission saved: {OUT/'submission.csv'}")

[22:03:53] [9/10] test ensemble
[22:03:53] [9/10] submission saved: /kaggle/working/queuewatch_v5_1_output/submission.csv


In [12]:
log("[10/10] final summary/package")
flags=[]
for f in range(FOLDS):
    ck=torch.load(MOD/f"queuewatch_fold{f}_best.pt",map_location="cpu"); flags.append(bool(ck.get("pretrained_loaded",False)))
summary={"project":"9% — QueueWatch","version":"v5.1 pretrained 5-fold count regression","labeled_images":int(len(manifest)),"test_images":int(len(test_images)),"all_labeled_images_used_across_cv":True,"model":"MobileNetV3-Small","target":"log1p(crowd_count)","image_size":IMG,"imagenet_pretrained_loaded_all_folds":bool(all(flags)),"oof_mae":float(oof_mae),"oof_rmse":float(oof_rmse),"naive_fold_median_mae":float(base_mae),"naive_fold_median_rmse":float(base_rmse),"mae_improvement_vs_naive":float(base_mae-oof_mae),"test_prediction":"median ensemble of 5 fold models","scope_note":"General crowd counting dataset, not queue-specific.","evaluation_note":"OOF predictions are generated only from models that did not train on their corresponding validation image."}
json.dump(summary,open(OUT/"final_summary.json","w"),indent=2)
print(json.dumps(summary,indent=2),flush=True)
zip_path=shutil.make_archive("/kaggle/working/queuewatch_9pct_v5_1_PRETRAINED_CV_FIXED_outputs","zip",root_dir=OUT)
log(f"[10/10] ZIP={zip_path}")
log("[DONE] QUEUEWATCH v5.1 PRETRAINED CV PIPELINE FINISHED")

[22:03:53] [10/10] final summary/package
{
  "project": "9% \u2014 QueueWatch",
  "version": "v5.1 pretrained 5-fold count regression",
  "labeled_images": 100,
  "test_images": 216,
  "all_labeled_images_used_across_cv": true,
  "model": "MobileNetV3-Small",
  "target": "log1p(crowd_count)",
  "image_size": 320,
  "imagenet_pretrained_loaded_all_folds": true,
  "oof_mae": 41.80534362792969,
  "oof_rmse": 65.5135139327185,
  "naive_fold_median_mae": 74.26000213623047,
  "naive_fold_median_rmse": 111.62419075154588,
  "mae_improvement_vs_naive": 32.45465850830078,
  "test_prediction": "median ensemble of 5 fold models",
  "scope_note": "General crowd counting dataset, not queue-specific.",
  "evaluation_note": "OOF predictions are generated only from models that did not train on their corresponding validation image."
}
[22:03:55] [10/10] ZIP=/kaggle/working/queuewatch_9pct_v5_1_PRETRAINED_CV_FIXED_outputs.zip
[22:03:55] [DONE] QUEUEWATCH v5.1 PRETRAINED CV PIPELINE FINISHED


## Final output

`/kaggle/working/queuewatch_9pct_v5_1_PRETRAINED_CV_FIXED_outputs.zip`

Success is only confirmed when:

`[DONE] QUEUEWATCH v5.1 PRETRAINED CV PIPELINE FINISHED`

Compare **OOF MAE** against **naive fold-median MAE**. If the model does not beat the naive baseline, do not overclaim the result.